# 02 - GPT Baseline Architecture (TinyLlama Reproduction)

In this notebook we implement a **decoder-only GPT architecture from scratch in PyTorch**.

This serves as the baseline model before introducing the TinyLlama improvements:

- RMSNorm
- Rotary Positional Embeddings (RoPE)
- SwiGLU
- Grouped Query Attention (GQA)

### Objectives

By the end of this notebook we will have:

- Token Embeddings
- Positional Embeddings
- Multi-Head Self Attention
- Feed Forward Network
- Decoder Blocks
- Complete GPT Model
- Shape Verification

No training will be performed in this notebook.

#Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Project Root
ROOT = Path("/content/drive/MyDrive/TinyLlama-repo")

# Required directories for this notebook
CONFIG_DIR = ROOT / "config"
CHECKPOINTS_DIR = ROOT / "checkpoints"
OUTPUTS_DIR = ROOT / "outputs"

# Create only the new directory introduced in this notebook
CONFIG_DIR.mkdir(exist_ok=True)

SRC_DIR = ROOT / "src"
SRC_DIR.mkdir(exist_ok=True)

print("Google Drive mounted successfully.")

import json

with open(CONFIG_DIR / "tokenizer_config.json", "r") as f:
    tokenizer_config = json.load(f)

print("Loaded tokenizer_config:", tokenizer_config)

Mounted at /content/drive
Google Drive mounted successfully.
Loaded tokenizer_config: {'vocab_size': 4096, 'pad_token_id': 1, 'bos_token_id': 2, 'eos_token_id': 3, 'max_seq_len': 512}


#Imports

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

print(torch.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", device)

2.11.0+cu128
Using: cuda


#Configuration

In [ ]:
@dataclass
class GPTConfig:

    vocab_size = tokenizer_config["vocab_size"]

    max_seq_len = 512

    d_model = 512

    n_heads = 8

    n_layers = 6

    dropout = 0.1

    bias = False

In [ ]:
config = GPTConfig()

# Token Embeddings

A language model cannot process words directly.

Each token ID is converted into a dense vector called an **embedding**.

For example

Token IDs

```
[15, 928, 54]
```

↓

Embeddings

```
[
 [0.12 ...],
 [0.83 ...],
 [-1.2 ...]
]
```

Each embedding has `d_model` dimensions.

#Token Embedding
Only one layer

In [ ]:
class TokenEmbedding(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.embedding = nn.Embedding(
            config.vocab_size,
            config.d_model
        )

    def forward(self, x):

        return self.embedding(x)

#Test Token Embedding
Expected:
Input Shape : torch.Size([2,10])

Output Shape: torch.Size([2,10,512])

In [ ]:
embedding = TokenEmbedding(config)

x = torch.randint(
    0,
    config.vocab_size,
    (2, 10)
)

out = embedding(x)

print("Input Shape :", x.shape)
print("Output Shape:", out.shape)

Input Shape : torch.Size([2, 10])
Output Shape: torch.Size([2, 10, 512])


# Positional Embeddings

The embedding layer knows **what** the token is.

It does **not** know **where** it appears.

For the baseline GPT we use **learned positional embeddings**.

Later, TinyLlama replaces this with **RoPE**.

#Positional Embedding

In [ ]:
class PositionalEmbedding(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.embedding = nn.Embedding(
            config.max_seq_len,
            config.d_model
        )

    def forward(self, x):

        batch_size, seq_len = x.shape

        positions = torch.arange(
            seq_len,
            device=x.device
        )

        positions = positions.unsqueeze(0)

        return self.embedding(positions)

Notice:
Thie input is
```
(batch, sequence)
```
but the output is
```
(1, sequence, embedding)
```
because every sample shares the same positions.

#Test Positional Embedding
Expected:
```
torch.Size([1,32,512])
```

In [ ]:
position = PositionalEmbedding(config)

x = torch.randint(
    0,
    config.vocab_size,
    (4, 32)
)

out = position(x)

print(out.shape)

torch.Size([1, 32, 512])


#Combine Both, Token Embeddings and Positional Embeddings

In [ ]:
token = TokenEmbedding(config)

position = PositionalEmbedding(config)

x = torch.randint(
    0,
    config.vocab_size,
    (2, 20)
)

token_embeddings = token(x)

position_embeddings = position(x)

hidden = token_embeddings + position_embeddings

print(hidden.shape)

torch.Size([2, 20, 512])


# Causal Self-Attention

Self-attention allows every token to decide **which previous tokens are important** when computing its representation.

For example:

```
The cat sat on the mat
```

When processing **"mat"**, the model might attend to:

- "The"
- "cat"
- "sat"

but **never future tokens**.

To prevent looking into the future we use a **causal mask**.

#Casual Mask

In [ ]:
def create_causal_mask(seq_len):

    mask = torch.tril(torch.ones(seq_len, seq_len))

    return mask

#Visualize Mask

In [ ]:
mask = create_causal_mask(8)

print(mask)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])


Expected:
```
1 0 0 0 0 0 0 0
1 1 0 0 0 0 0 0
1 1 1 0 0 0 0 0
1 1 1 1 0 0 0 0
...
```
Every token can only see itself and previous tokens.

## Multi-Head Attention

Every attention head learns something different.

One head may learn grammar.

Another may learn long-range dependencies.

Another may learn punctuation.

Each head computes

- Query (Q)
- Key (K)
- Value (V)

using simple linear layers.

#QKV Projection

In [ ]:
class QKVProjection(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.query = nn.Linear(
            config.d_model,
            config.d_model,
            bias=config.bias
        )

        self.key = nn.Linear(
            config.d_model,
            config.d_model,
            bias=config.bias
        )

        self.value = nn.Linear(
            config.d_model,
            config.d_model,
            bias=config.bias
        )

    def forward(self, x):

        q = self.query(x)

        k = self.key(x)

        v = self.value(x)

        return q, k, v

#Test QKV

In [ ]:
projection = QKVProjection(config)

x = torch.randn(2, 16, config.d_model)

q, k, v = projection(x)

print(q.shape)
print(k.shape)
print(v.shape)

torch.Size([2, 16, 512])
torch.Size([2, 16, 512])
torch.Size([2, 16, 512])


Expected:
```
torch.Size([2,16,512])
torch.Size([2,16,512])
torch.Size([2,16,512])
```

## Split into Multiple Heads

Instead of one large attention operation, GPT splits the embedding into multiple smaller attention heads.

Example:

Embedding Dimension = 512

Heads = 8

Each head gets

```
512 / 8 = 64
```

dimensions.

#Split Heads

In [ ]:
def split_heads(x, config):

    batch_size, seq_len, d_model = x.shape

    head_dim = d_model // config.n_heads

    x = x.view(
        batch_size,
        seq_len,
        config.n_heads,
        head_dim
    )

    x = x.transpose(1, 2)

    return x

#Test

In [ ]:
x = torch.randn(2, 16, config.d_model)

heads = split_heads(x, config)

print(heads.shape)

torch.Size([2, 8, 16, 64])


Expected:
```
torch.Size([2,8,16,64])
```
Meaning:
```
Batch = 2

Heads = 8

Sequence =16

Head Dimension =64
```

#Merge Heads
Later we'll combine them back.

In [ ]:
def merge_heads(x):

    batch_size, heads, seq_len, head_dim = x.shape

    x = x.transpose(1, 2)

    x = x.contiguous()

    x = x.view(
        batch_size,
        seq_len,
        heads * head_dim
    )

    return x

#Test Merge

In [ ]:
def merge_heads(x):

    batch_size, heads, seq_len, head_dim = x.shape

    x = x.transpose(1, 2)

    x = x.contiguous()

    x = x.view(
        batch_size,
        seq_len,
        heads * head_dim
    )

    return x

Expected:
```
torch.Size([2,16,512])
```
Exactly the original shape.

## Scaled Dot Product Attention

Now we compute

```
Attention = Softmax(QKᵀ / √d)
```

where

- Q = Query
- K = Key
- V = Value

The result is then multiplied with the Value vectors.

This is the core computation of a Transformer.

#Scaled Dot Product Attention

In [ ]:
class ScaledDotProductAttention(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.scale = math.sqrt(
            config.d_model // config.n_heads
        )

    def forward(self, q, k, v):

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        )

        scores = scores / self.scale

        seq_len = scores.size(-1)

        mask = create_causal_mask(seq_len).to(scores.device)

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )

        attention = F.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention,
            v
        )

        return output

#Test Attention

In [ ]:
attention = ScaledDotProductAttention(config)

x = torch.randn(2, 16, config.d_model)

projection = QKVProjection(config)

q, k, v = projection(x)

q = split_heads(q, config)
k = split_heads(k, config)
v = split_heads(v, config)

out = attention(q, k, v)

print(out.shape)

torch.Size([2, 8, 16, 64])


Expected:
```
torch.Size([2,8,16,64])
```

#Output Projection
After merging the heads, GPT applies one final linear layer.

In [ ]:
class OutputProjection(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.proj = nn.Linear(
            config.d_model,
            config.d_model,
            bias=config.bias
        )

    def forward(self, x):

        return self.proj(x)

#Complete Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.qkv = QKVProjection(config)

        self.attention = ScaledDotProductAttention(config)

        self.output = OutputProjection(config)

        self.config = config

    def forward(self, x):

        q, k, v = self.qkv(x)

        q = split_heads(q, self.config)
        k = split_heads(k, self.config)
        v = split_heads(v, self.config)

        out = self.attention(q, k, v)

        out = merge_heads(out)

        out = self.output(out)

        return out

#Final Test

In [ ]:
mha = MultiHeadAttention(config)

x = torch.randn(2, 32, config.d_model)

out = mha(x)

print(out.shape)

torch.Size([2, 32, 512])


Expected:
```
torch.Size([2,32,512])

# Feed Forward Network (FFN)

After the attention mechanism, every token independently passes through a small neural network called the Feed Forward Network (FFN).

In the baseline GPT, this consists of:

```
Linear
   ↓
GELU
   ↓
Linear
```

The hidden dimension is typically **4 × d_model**.

Later, TinyLlama replaces this entire block with **SwiGLU**.

#Feed Forward Network

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                config.d_model,
                4 * config.d_model,
                bias=config.bias
            ),

            nn.GELU(),

            nn.Linear(
                4 * config.d_model,
                config.d_model,
                bias=config.bias
            ),

            nn.Dropout(config.dropout)
        )

    def forward(self, x):

        return self.net(x)

#Test FFN

In [ ]:
ffn = FeedForward(config)

x = torch.randn(2, 16, config.d_model)

out = ffn(x)

print(out.shape)

torch.Size([2, 16, 512])


Expected
```
torch.Size([2,16,512])

# Decoder Block

One decoder block contains

```
Input
   │
LayerNorm
   │
Multi Head Attention
   │
Residual
   │
LayerNorm
   │
Feed Forward Network
   │
Residual
```

This block is repeated multiple times to build the model.

#Decoder Block

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.ln1 = nn.LayerNorm(config.d_model)

        self.attention = MultiHeadAttention(config)

        self.ln2 = nn.LayerNorm(config.d_model)

        self.ffn = FeedForward(config)

    def forward(self, x):

        x = x + self.attention(self.ln1(x))

        x = x + self.ffn(self.ln2(x))

        return x

#Test Decoder Block

In [ ]:
block = DecoderBlock(config)

x = torch.randn(2, 32, config.d_model)

out = block(x)

print(out.shape)

torch.Size([2, 32, 512])


Expected
```
torch.Size([2,32,512])

# GPT Model

Now we assemble everything together.

```
Token Embedding
        +
Position Embedding
        │
        ▼
Decoder Block × N
        │
LayerNorm
        │
Linear Head
        │
Vocabulary Logits
```

This is the complete decoder-only GPT architecture.

#GPT Model

In [ ]:
class GPT(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.token_embedding = TokenEmbedding(config)

        self.position_embedding = PositionalEmbedding(config)

        self.blocks = nn.ModuleList(

            [
                DecoderBlock(config)
                for _ in range(config.n_layers)
            ]

        )

        self.final_norm = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False
        )

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        x = x + self.position_embedding(input_ids)

        for block in self.blocks:

            x = block(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

#Create Model

In [ ]:
model = GPT(config).to(device)

print(model)

GPT(
  (token_embedding): TokenEmbedding(
    (embedding): Embedding(4096, 512)
  )
  (position_embedding): PositionalEmbedding(
    (embedding): Embedding(512, 512)
  )
  (blocks): ModuleList(
    (0-5): 6 x DecoderBlock(
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attention): MultiHeadAttention(
        (qkv): QKVProjection(
          (query): Linear(in_features=512, out_features=512, bias=False)
          (key): Linear(in_features=512, out_features=512, bias=False)
          (value): Linear(in_features=512, out_features=512, bias=False)
        )
        (attention): ScaledDotProductAttention()
        (output): OutputProjection(
          (proj): Linear(in_features=512, out_features=512, bias=False)
        )
      )
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=False)
          (1): GELU(approximate='none')
         

#Parameter Count
This gives you an idea of the model size.

In [ ]:
num_params = sum(

    p.numel()

    for p in model.parameters()

)

print(f"Parameters: {num_params:,}")

Parameters: 23,344,128


#Forward Pass

In [ ]:
x = torch.randint(

    0,

    config.vocab_size,

    (2, 128)

).to(device)

with torch.no_grad():

    logits = model(x)

print(logits.shape)

torch.Size([2, 128, 4096])


Expected
```
torch.Size([2,128,32000])

#Unit Test

In [ ]:
assert logits.shape == (

    2,

    128,

    config.vocab_size

)

print("Forward pass successful!")

Forward pass successful!


#Save Model Config

In [ ]:
from pathlib import Path
import json

CONFIG_DIR = Path("/content/drive/MyDrive/TinyLlama-repo/config")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

config_dict = {
    "vocab_size": config.vocab_size,
    "max_seq_len": config.max_seq_len,
    "d_model": config.d_model,
    "n_heads": config.n_heads,
    "n_layers": config.n_layers,
    "dropout": config.dropout,
    "bias": config.bias
}

with open(CONFIG_DIR / "gpt_config.json", "w") as f:
    json.dump(config_dict, f, indent=4)

print("Configuration saved.")

Configuration saved.


#Architecture Summary

In [ ]:
print("=" * 50)
print("Baseline GPT Completed")
print("=" * 50)
print(f"Embedding Dimension : {config.d_model}")
print(f"Attention Heads     : {config.n_heads}")
print(f"Decoder Layers      : {config.n_layers}")
print(f"Vocabulary Size     : {config.vocab_size}")
print(f"Parameters          : {num_params:,}")
print("=" * 50)

Baseline GPT Completed
Embedding Dimension : 512
Attention Heads     : 8
Decoder Layers      : 6
Vocabulary Size     : 4096
Parameters          : 23,344,128


In [ ]:
from pathlib import Path

gpt_code = r'''
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class TokenEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.embedding = nn.Embedding(
            config.vocab_size,
            config.d_model
        )

    def forward(self, x):
        return self.embedding(x)


class PositionalEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.embedding = nn.Embedding(
            config.max_seq_len,
            config.d_model
        )

    def forward(self, x):

        _, seq_len = x.shape

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0)

        return self.embedding(positions)


def create_causal_mask(seq_len):

    return torch.tril(torch.ones(seq_len, seq_len))


def split_heads(x, n_heads):

    batch_size, seq_len, d_model = x.shape

    head_dim = d_model // n_heads

    x = x.view(
        batch_size,
        seq_len,
        n_heads,
        head_dim
    )

    return x.transpose(1,2)


def merge_heads(x):

    batch_size, heads, seq_len, head_dim = x.shape

    x = x.transpose(1,2).contiguous()

    return x.view(
        batch_size,
        seq_len,
        heads * head_dim
    )


class MultiHeadAttention(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.q = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v = nn.Linear(config.d_model, config.d_model, bias=config.bias)

        self.out = nn.Linear(config.d_model, config.d_model, bias=config.bias)

    def forward(self,x):

        q = split_heads(self.q(x), self.n_heads)
        k = split_heads(self.k(x), self.n_heads)
        v = split_heads(self.v(x), self.n_heads)

        scores = torch.matmul(
            q,
            k.transpose(-2,-1)
        )

        scores = scores / math.sqrt(self.head_dim)

        mask = create_causal_mask(
            scores.size(-1)
        ).to(scores.device)

        scores = scores.masked_fill(
            mask==0,
            float("-inf")
        )

        weights = F.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(weights,v)

        out = merge_heads(out)

        return self.out(out)


class FeedForward(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                config.d_model,
                4*config.d_model,
                bias=config.bias
            ),
            nn.GELU(),
            nn.Linear(
                4*config.d_model,
                config.d_model,
                bias=config.bias
            ),
            nn.Dropout(config.dropout)
        )

    def forward(self,x):

        return self.net(x)


class DecoderBlock(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.ln1 = nn.LayerNorm(config.d_model)

        self.attn = MultiHeadAttention(config)

        self.ln2 = nn.LayerNorm(config.d_model)

        self.ffn = FeedForward(config)

    def forward(self,x):

        x = x + self.attn(self.ln1(x))

        x = x + self.ffn(self.ln2(x))

        return x


class GPT(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.token_embedding = TokenEmbedding(config)

        self.position_embedding = PositionalEmbedding(config)

        self.blocks = nn.ModuleList(
            [
                DecoderBlock(config)
                for _ in range(config.n_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False
        )

    def forward(self,input_ids):

        x = self.token_embedding(input_ids)

        x = x + self.position_embedding(input_ids)

        for block in self.blocks:

            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)
'''

with open(SRC_DIR / "gpt_baseline.py", "w") as f:
    f.write(gpt_code)

print("gpt_baseline.py saved successfully.")

gpt_baseline.py saved successfully.


#Save Baseline Checkpoint (resume-safe, fp16)

In [ ]:
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Re-import the model from the file we just wrote, so the checkpoint
# always matches what later notebooks (src.gpt_baseline.GPT) will load.
import importlib, sys
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import src.gpt_baseline as gpt_baseline_module
importlib.reload(gpt_baseline_module)

model_final = gpt_baseline_module.GPT(config).to(device)

baseline_ckpt = CHECKPOINTS_DIR / "gpt_baseline.pt"
current_keys = set(model_final.state_dict().keys())

def _checkpoint_keys_match(path):
    try:
        ck = torch.load(path, map_location="cpu")
        return set(ck.keys()) == current_keys
    except Exception:
        return False

if baseline_ckpt.exists() and _checkpoint_keys_match(baseline_ckpt):
    print("Checkpoint exists and matches current model, skip:", baseline_ckpt)
else:
    if baseline_ckpt.exists():
        print("Checkpoint exists but keys mismatch current model -- overwriting.")
    fp16_state = {k: v.half() for k, v in model_final.state_dict().items()}
    torch.save(fp16_state, baseline_ckpt)
    print("Saved:", baseline_ckpt)

Saved: /content/drive/MyDrive/TinyLlama-repo/checkpoints/gpt_baseline.pt
